In [1]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import seaborn as sns

import numpy as np
import pandas as pd

In [2]:
df=pd.read_csv("preprocessed_data.csv")

In [3]:
df.head()

,Date,Open,High,Low,Close,Adj Close,Volume,Year,Month,Day,DayOfWeek,IsWeekend
0,2018-02-05,-1.447772,-1.441465,-1.510141,-1.522047,-1.522047,1.062268,2018,2,5,0,0.0
1,2018-02-06,-1.579589,-1.452453,-1.556931,-1.416167,-1.416167,1.156404,2018,2,6,1,0.0
2,2018-02-07,-1.405553,-1.399802,-1.377121,-1.426885,-1.426885,0.599365,2018,2,7,2,0.0
3,2018-02-08,-1.400944,-1.444029,-1.510420,-1.560481,-1.560481,0.657948,2018,2,8,3,0.0
4,2018-02-09,-1.522898,-1.552262,-1.639627,-1.566302,-1.566302,1.641237,2018,2,9,4,0.0


# Prepare the Dataset

In [4]:
# Assuming your DataFrame is df, and you have already done preprocessing
features = ['Open', 'High', 'Low', 'Volume']  # Add more features if needed
target = 'Close'

X = df[features]
y = df[target]


#  Scale the Features

In [5]:
# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# Initialize the Random Forest Regressor Model

In [6]:
# Initialize the Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, random_state=42)


#  Perform K-fold Cross-Validation

In [7]:
# Perform K-fold cross-validation (for instance, 5-fold cross-validation)
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Use cross_val_score to evaluate the model with cross-validation
cv_scores = cross_val_score(model, X_scaled, y, cv=cv, scoring='neg_mean_squared_error')

# Since scoring='neg_mean_squared_error', the returned values are negative, so we convert them to positive
cv_scores = -cv_scores

# Print cross-validation results
print(f"Cross-validation MSE scores: {cv_scores}")
print(f"Mean MSE: {np.mean(cv_scores)}")
print(f"Standard Deviation of MSE: {np.std(cv_scores)}")


Cross-validation MSE scores: [0.00225881 0.00276369 0.00266764 0.00229468 0.00221677]
Mean MSE: 0.002440315866801754
Standard Deviation of MSE: 0.0002281985554064048


# Hyperparameter Tuning with Grid Search

In [8]:


param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
}

# Perform Grid Search with Cross-validation
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)

grid_search.fit(X_scaled, y)

# Print best parameters from Grid Search
print("Best parameters:", grid_search.best_params_)

# Get the best model
best_model = grid_search.best_estimator_


Best parameters: {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}


# Evaluate the Best Model on the Test Data

In [9]:

# Split the dataset into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Train the best model
best_model.fit(X_train,)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Evaluate the performance
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse}")
print(f"R-squared: {r2}")


TypeError: BaseForest.fit() missing 1 required positional argument: 'y'

# Visualize the Model's Predictions vs. Actual Values

In [ ]:


# Plot the actual vs predicted values
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, color='blue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', linewidth=2)
plt.title('Actual vs Predicted')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.show()


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Initialize and train the Random Forest Regressor
regressor = RandomForestRegressor(n_estimators=100, random_state=42)
regressor.fit(X_train, y_train)

# Make predictions on the test set
y_pred = regressor.predict(X_test)

# Evaluate the model using regression metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae}")
print(f"Mean Squared Error (MSE): {mse}")
print(f"Root Mean Squared Error (RMSE): {rmse}")
print(f"R² Score: {r2}")
